# Leetcode 1084
## Sales Analysis III

In [0]:
# https://leetcode.com/problems/sales-analysis-iii
# Completed at 2026/04/21

In [0]:
# Frameworks
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
import pyspark.sql.functions as F
from datetime import datetime

## Part 1: Parsing the raw data into PySpark Dataframes

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime


def parse_raw_to_df(table_str):
    spark = SparkSession.builder.getOrCreate()

    # Step 1 — clean lines
    lines = []

    for line in table_str.splitlines():
        line = line.strip()

        if not line:
            continue

        if line.startswith("+"):
            continue

        lines.append(line)

    # Step 2 — extract schema
    schema_cols = []
    schema_types = []

    i = 0

    # find schema header
    while i < len(lines):
        if "Column Name" in lines[i]:
            i += 1
            break
        i += 1

    # read schema rows
    while i < len(lines):
        parts = [p.strip() for p in lines[i].split("|") if p.strip()]

        if len(parts) == 2:
            schema_cols.append(parts[0])
            schema_types.append(parts[1])
        else:
            break

        i += 1

    # Step 3 — Spark type mapping
    type_map = {
        "int": IntegerType(),
        "integer": IntegerType(),
        "string": StringType(),
        "varchar": StringType(),
        "char": StringType(),
        "date": DateType()
    }

    fields = []

    for col, typ in zip(schema_cols, schema_types):
        spark_type = type_map.get(typ.lower(), StringType())

        fields.append(
            StructField(col, spark_type, True)
        )

    schema = StructType(fields)

    # Step 4 — find data header
    data_start = None

    for idx, line in enumerate(lines):

        parts = [p.strip() for p in line.split("|") if p.strip()]

        if parts == schema_cols:
            data_start = idx + 1
            break

    if data_start is None:
        raise ValueError("Data section not found")

    # Step 5 — parse data rows
    data_rows = []

    for line in lines[data_start:]:

        parts = [p.strip() for p in line.split("|") if p.strip()]

        if len(parts) != len(schema_cols):
            continue

        converted_row = []

        for val, typ in zip(parts, schema_types):

            typ = typ.lower()

            if typ in ["int", "integer"]:
                converted_row.append(int(val))

            elif typ == "date":
                converted_row.append(
                    datetime.strptime(val, "%Y-%m-%d").date()
                )

            else:
                converted_row.append(val)

        data_rows.append(tuple(converted_row))

    # Step 6 — create dataframe
    df = spark.createDataFrame(data_rows, schema)

    return df

In [0]:
# Initial data tables

# EMPLOYEE
r_product = """
+--------------+---------+
| Column Name  | Type    |
+--------------+---------+
| product_id   | int     |
| product_name | varchar |
| unit_price   | int     |
+--------------+---------+
+------------+--------------+------------+
| product_id | product_name | unit_price |
+------------+--------------+------------+
| 1          | S8           | 1000       |
| 2          | G4           | 800        |
| 3          | iPhone       | 1400       |
+------------+--------------+------------+
"""
r_sales = """
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| seller_id   | int     |
| product_id  | int     |
| buyer_id    | int     |
| sale_date   | date    |
| quantity    | int     |
| price       | int     |
+-------------+---------+
+-----------+------------+----------+------------+----------+-------+
| seller_id | product_id | buyer_id | sale_date  | quantity | price |
+-----------+------------+----------+------------+----------+-------+
| 1         | 1          | 1        | 2019-01-21 | 2        | 2000  |
| 1         | 2          | 2        | 2019-02-17 | 1        | 800   |
| 2         | 2          | 3        | 2019-06-02 | 1        | 800   |
| 3         | 3          | 4        | 2019-05-13 | 2        | 2800  |
+-----------+------------+----------+------------+----------+-------+
"""
product_df = parse_raw_to_df(r_product)
product_df.createOrReplaceTempView("product")

sales_df = parse_raw_to_df(r_sales)
sales_df.createOrReplaceTempView("sales")

In [0]:
product_df.printSchema()
sales_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- unit_price: integer (nullable = true)

root
 |-- seller_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- buyer_id: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)



## Part 2: Querying and Solution

Write a solution to report the products that were only sold in the first quarter of 2019. That is, between 2019-01-01 and 2019-03-31 inclusive.

Return the result table in any order.

The result format is in the following example.

![image_1776761162299.png](./image_1776761162299.png "image_1776761162299.png")

### SparkSQL

In [0]:
sol_sql = spark.sql("""
                        SELECT p.product_id, p.product_name
                        FROM Product p
                        JOIN sales s
                        ON s.product_id = p.product_id
                        GROUP BY p.product_id, p.product_name
                        HAVING min(sale_date) >= '2019-01-01' AND max(sale_date) <= '2019-03-31'
                    """).show()

+----------+------------+
|product_id|product_name|
+----------+------------+
|         1|          S8|
+----------+------------+



In [0]:
%sql
SELECT p.product_id, p.product_name
FROM Product p
JOIN sales s
ON s.product_id = p.product_id
GROUP BY p.product_id, p.product_name
HAVING min(sale_date) >= '2019-01-01' AND max(sale_date) <= '2019-03-31'

product_id,product_name
1,S8


### PySpark


In [0]:
# Pyspark code
joined_sales = product_df.join(sales_df, "product_id")
grouped = joined_sales.groupBy("product_id", "product_name").agg(
    F.min("sale_date").alias("min_date"),
    F.max("sale_date").alias("max_date")
).filter(
    (F.col("min_date") >= "2019-01-01") & (F.col("max_date") <= "2019-03-31")
)
grouped.select("product_id", "product_name").show()

+----------+------------+
|product_id|product_name|
+----------+------------+
|         1|          S8|
+----------+------------+

